In [ ]:
import importlib
import weights_cuda
import spike_engine_cuda
importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)
from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus
import cupy as cp


In [ ]:
# N = 50, lifetime = 100000 # can it finish in a minute?
N = 512
lifetime = 50000
record_stride = 11
engine = SpikeEngineCUDA(
    square_torus(N),
    (N, N),
    use_k2tree=True,
    verify_k2tree=False,
    verify_progress_every=2000,
    decay_rate=0.949,
    rank=64,
    #weight_initializer=lambda size: cp.zeros(size)
)


In [ ]:
# Static wave-propagation benchmark only: this intentionally uses constant weights.
# The reservoir notebooks use heterogeneous random weights scaled near bifurcation instead.
w_accum, w_instant = engine.estimate_bifurcation_weight(input_period=1)
target, _, _ = engine.set_constant_weights_near_bifurcation(input_period=1, scale=1.052, freeze_learning=True)
print(f"w_accum={w_accum:.6f} w_instant={w_instant:.6f} target={target:.6f}")
print("constant weights are enabled for this static propagation benchmark only")


In [ ]:
input_neuron = (N * N) // 2 + N//2
engine.set_input_neurons([input_neuron])
inputspikes = cp.ones((lifetime, 1), dtype=cp.float32)
recorded_frames = (lifetime + record_stride - 1) // record_stride
estimated_bytes = N * N * recorded_frames * cp.dtype(cp.float32).itemsize
print(f"streaming {recorded_frames} frames, uncompressed membrane payload {spike_engine_cuda._format_bytes(estimated_bytes)}")

engine.start_static_record(
    inputspikes,
    lifetime,
    "cuda_test_10.spire.gz",
    record_membrane=True,
    full_decay=True,
    compression_level=4,
    compression_async=True,
    record_stride=record_stride,
)
# rsync -avP user@remote:/path/to/file /local/path


In [ ]:
# Optional: validate neighbors for a random neuron
idx = 1000
print("neighbors:", engine.weights.get_neighbors(idx))


In [ ]:
recorded_frames = (lifetime + record_stride - 1) // record_stride
estimated_bytes = N * N * recorded_frames * cp.dtype(cp.float32).itemsize
print(spike_engine_cuda._format_bytes(estimated_bytes))
